# Counterfactual degradation-order audit for A0-L

This notebook evaluates whether the frozen A0-L baseline is sensitive to the **formation order** of the same degradation factors. Every permutation reuses exactly the same masks, noise and parameters. Results include raw inputs and a secondary severity-matched control. The notebook never loads `CDD-11_test` and never trains a new proposed method. Because A0-L checkpoint selection already used these validation scenes, this is hypothesis discovery rather than an unbiased final performance claim.

If an exact A0-L checkpoint is attached as a Kaggle input it is reused; otherwise the approved 20-epoch A0-L baseline is reproduced first. Expected end-to-end runtime is roughly 40–65 minutes on 2xT4, depending on whether reproduction is needed.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_order_audit")
A0_RUN_NAME = "a0l_nafnet_baseline_sidd32_seed42_20ep"
A0_CONFIG = Path("configs/calibration_controls_long.json")
AUDIT_DIR = Path("/kaggle/working/degradation_order_audit")

assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert A0_CONFIG.is_file(), f"Missing config: {A0_CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
)
print("GPUs:", gpu_names)
print("CDD-11:", CDD11_ROOT)

In [ ]:
# Read-only verification before reproducing or loading A0-L.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_order.json",
], check=True)

In [ ]:
# Reuse an explicitly attached exact A0-L checkpoint when available.
attached_candidates = sorted(
    path for path in Path("/kaggle/input").rglob("best.pt")
    if path.parent.name == A0_RUN_NAME or path.parent.parent.name == A0_RUN_NAME
)
local_checkpoint = EXPERIMENTS_ROOT / A0_RUN_NAME / "best.pt"
if attached_candidates:
    A0_CHECKPOINT = attached_candidates[0]
    print("Using attached A0-L checkpoint:", A0_CHECKPOINT)
else:
    if not local_checkpoint.is_file():
        subprocess.run([
            "python", "-m", "hybrid_cot_nafnet.run_ablation",
            "--config", str(A0_CONFIG),
            "--data-root", str(CDD11_ROOT),
            "--experiments-root", str(EXPERIMENTS_ROOT),
            "--nproc-per-node", "2",
            "--runs", A0_RUN_NAME,
        ], check=True)
    A0_CHECKPOINT = local_checkpoint
    print("Using reproduced A0-L checkpoint:", A0_CHECKPOINT)
assert A0_CHECKPOINT.is_file(), f"Missing A0-L checkpoint: {A0_CHECKPOINT}"

In [ ]:
# Approved diagnostic: frozen model, validation scenes only, no test data.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_degradation_order",
    "--checkpoint", str(A0_CHECKPOINT),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(AUDIT_DIR),
    "--realizations", "3",
    "--generation-seed", "20260920",
    "--tile", "0",
    "--save-panels", "1",
], check=True)

In [ ]:
import pandas as pd
from IPython.display import display

summary = json.loads((AUDIT_DIR / "summary.json").read_text())
protocol = json.loads((AUDIT_DIR / "protocol.json").read_text())
print(json.dumps(summary, indent=2))
print("Validation scenes:", protocol["scene_ids"])
order_table = pd.read_csv(AUDIT_DIR / "per_order.csv")
display(order_table[order_table["mode"] == "severity_matched"].sort_values(
    ["material_consistent_order_effect", "mean_delta_gain_psnr_vs_canonical"],
    ascending=[False, True],
))

In [ ]:
import zipfile
from IPython.display import FileLink

required = ["summary.json", "protocol.json", "per_sample.csv", "per_order.csv"]
for name in required:
    assert (AUDIT_DIR / name).is_file(), f"Missing audit artifact: {name}"
output_zip = Path("/kaggle/working/degradation_order_audit_results.zip")
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(AUDIT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(AUDIT_DIR))
print("Download this ZIP and place it under results/:")
display(FileLink(str(output_zip)))